# Train All 5 Fixes — TB CXR Pipeline

Run this notebook on a **Kaggle T4 GPU** session (16 GB, ~9 h limit).  
All cells run sequentially; estimated total wall-clock: **5–6 hours**.

## Datasets to attach before running
| Kaggle dataset slug | Mount path | Purpose |
|---|---|---|
| `iahmedhabib/montgomery` | `/kaggle/input/montgomery` | Fix 1 & 2: TB labels + DANN |
| `iahmedhabib/shehzhenn` | `/kaggle/input/shehzhenn` | Fix 1 & 2: TB labels + DANN |
| `usmanshams/tbx-11` | `/kaggle/input/tbx-11` | Fix 1 & 2 & 5: TB labels + MoE |
| `organizations/nih-chest-xrays` | `/kaggle/input/nih-chest-xrays` | Fix 2: DANN domain |
| `iahmedhabib/medsam-vit-b` | `/kaggle/input/medsam-vit-b` | All fixes: backbone |

## What each cell trains
1. **Setup** — clone repo, install deps, discover dataset roots  
2. **Fix 2 — DANN retrain** (~70 min): Component 1 with corrected λ=0.3, 20 epochs  
3. **Fix 1 & 3 — TXV + TB head** (~40 min): Component 2 with binary TB BCE loss  
4. **Fix 4 — MedSAM augmentation** (~90 min): Component 4 with flip + brightness jitter  
5. **Fix 5 — MoE cache + expert pretraining** (~60 min): generate CAM pseudo-masks, pretrain 4 experts  
6. **Collect outputs** — copies all checkpoints to `/kaggle/working/`


In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'
], check=True)

# ── Dataset root discovery ──────────────────────────────────────────────────
INPUT = Path('/kaggle/input')

def _find(candidates):
    for p in candidates:
        if Path(p).exists():
            return str(p)
    return None

MONTGOMERY_ROOT = _find([
    INPUT / 'montgomery' / 'montgomery',
    INPUT / 'montgomery',
    INPUT / 'datasets' / 'iahmedhabib' / 'montgomery' / 'montgomery',
])
SHENZHEN_ROOT = _find([
    INPUT / 'shehzhenn' / 'shenzhen',
    INPUT / 'shehzhenn',
    INPUT / 'datasets' / 'iahmedhabib' / 'shehzhenn' / 'shenzhen',
])
TBX11K_ROOT = _find([
    INPUT / 'tbx-11' / 'TBX11K',
    INPUT / 'tbx-11',
    INPUT / 'datasets' / 'usmanshams' / 'tbx-11' / 'TBX11K',
])
NIH_ROOT = _find([
    INPUT / 'nih-chest-xrays' / 'data',
    INPUT / 'nih-chest-xrays',
    INPUT / 'datasets' / 'organizations' / 'nih-chest-xrays' / 'data',
])
MEDSAM_CKPT = _find([
    INPUT / 'medsam-vit-b' / 'medsam_vit_b.pth',
    INPUT / 'datasets' / 'iahmedhabib' / 'medsam-vit-b' / 'medsam_vit_b.pth',
    REPO_DIR / 'checkpoints' / 'medsam' / 'medsam_vit_b.pth',
])

for name, val in [('MONTGOMERY_ROOT', MONTGOMERY_ROOT), ('SHENZHEN_ROOT', SHENZHEN_ROOT),
                   ('TBX11K_ROOT', TBX11K_ROOT), ('NIH_ROOT', NIH_ROOT), ('MEDSAM_CKPT', MEDSAM_CKPT)]:
    status = '✓' if val else '✗ NOT FOUND'
    print(f'{name}: {val or status}')

assert MEDSAM_CKPT, 'MedSAM checkpoint is required for all training steps'
assert any([MONTGOMERY_ROOT, SHENZHEN_ROOT, TBX11K_ROOT]), 'Need at least one labeled dataset'


In [ ]:
# ── Cell 2: Fix 2 — DANN Retrain (Component 1) ─────────────────────────────
# What this trains:  MedSAM ViT-B backbone with LoRA adapters (rank=4) + a
# 4-way domain classifier whose gradient is reversed by the GRL.
#
# What the pipeline learns:  The LoRA adapters must produce image embeddings
# that fool the domain classifier — i.e., features that look the same whether
# the image came from a Montgomery chest X-ray or a TBX11K scan.
#
# Key fix vs old run:  max_lambda 1.0 → 0.3 (gentler reversal) and epochs
# 8 → 20 (enough time for the LoRA to gradually adapt before reversal peaks).
#
# Expected result:  domain accuracy drops from 0.355 toward 0.25 (chance).
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
OUTPUT_C1 = Path('/kaggle/working/component1_runs')
OUTPUT_C1.mkdir(parents=True, exist_ok=True)

# Build an inline YAML that overrides only the paths relative to Kaggle
kaggle_c1_cfg = REPO_DIR / 'checkpoints' / 'component1' / 'component1_dann.kaggle.yaml'

# Patch dataset roots into environment so the config resolver picks them up
env = os.environ.copy()
if MONTGOMERY_ROOT: env['MONTGOMERY_ROOT'] = str(MONTGOMERY_ROOT)
if SHENZHEN_ROOT:   env['SHENZHEN_ROOT']   = str(SHENZHEN_ROOT)
if TBX11K_ROOT:     env['TBX11K_ROOT']     = str(TBX11K_ROOT)
if NIH_ROOT:        env['NIH_CXR14_ROOT']  = str(NIH_ROOT)
env['MEDSAM_VIT_B_CKPT'] = str(MEDSAM_CKPT)

cmd = [
    sys.executable, '-m', 'src.training.train_component1_dann',
    '--config', str(kaggle_c1_cfg),
    '--paths', str(REPO_DIR / 'checkpoints' / 'component1' / 'paths.kaggle.yaml'),
]
print('Starting DANN training (Fix 2)...')
print(' '.join(cmd))
result = subprocess.run(cmd, env=env, cwd=str(REPO_DIR))
print(f'DANN training exit code: {result.returncode}')
assert result.returncode == 0, 'DANN training failed'

# Verify outputs
for fname in ['component1_dann_full.pt', 'component1_adapters.safetensors']:
    p = OUTPUT_C1 / fname
    print(f'{fname}: {"found" if p.exists() else "MISSING"}')


In [ ]:
# ── Cell 3: Fix 1 & 3 — TXV + TB head (Component 2) ───────────────────────
# What this trains:  The domain routing head on top of frozen DenseNet121 AND
# a new single-neuron TB head (Linear(1024, 1)).
#
# What the pipeline learns (two things at once):
#   • Routing head — learns to cluster image embeddings by scanner/dataset so
#     the downstream system can adapt its priors by domain.
#   • TB head — learns to separate TB-positive from TB-negative by optimising
#     Binary Cross-Entropy on the Montgomery, Shenzhen, and TBX11K TB labels.
#     After training, sigmoid(tb_head(pooled)) ≈ P(TB | image).
#
# Fix 3 effect:  The TB head's weight vector [1024] doubles as a CAM projection.
# At inference time, the lesion proposer computes
#   CAM = relu(features_7x7 * tb_weight).sum() * sigmoid(tb_logit)
# instead of averaging over generic pathology classes.  The heatmap now lights
# up regions that the TB head found predictive — cavities, consolidations, and
# apical infiltrates — rather than random bright regions.
#
# Expected result:  Timika AUROC jumps from 0.296 (worse-than-chance) to 0.75+.
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
OUTPUT_C2 = Path('/kaggle/working/component2_runs')
OUTPUT_C2.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
if MONTGOMERY_ROOT: env['MONTGOMERY_ROOT'] = str(MONTGOMERY_ROOT)
if SHENZHEN_ROOT:   env['SHENZHEN_ROOT']   = str(SHENZHEN_ROOT)
if TBX11K_ROOT:     env['TBX11K_ROOT']     = str(TBX11K_ROOT)
if NIH_ROOT:        env['NIH_CXR14_ROOT']  = str(NIH_ROOT)
env['COMPONENT2_SAVE_DIR'] = str(OUTPUT_C2)

cmd = [
    sys.executable, '-m', 'src.training.train_component2_txv',
    '--config', str(REPO_DIR / 'configs' / 'component2_txv.kaggle.yaml'),
    '--paths', str(REPO_DIR / 'checkpoints' / 'component1' / 'paths.kaggle.yaml'),
    '--component1_config', str(REPO_DIR / 'checkpoints' / 'component1' / 'component1_dann.kaggle.yaml'),
]
print('Starting TXV + TB head training (Fix 1 & 3)...')
result = subprocess.run(cmd, env=env, cwd=str(REPO_DIR))
print(f'TXV training exit code: {result.returncode}')
assert result.returncode == 0, 'TXV training failed'

c2_ckpt = OUTPUT_C2 / 'component2_routing_head.pt'
print(f'Component 2 checkpoint: {"found" if c2_ckpt.exists() else "MISSING"}')


In [ ]:
# ── Cell 4: Fix 4 — MedSAM Augmentation (Component 4) ─────────────────────
# What this trains:  Only the MedSAM mask decoder (the ViT-B image encoder
# and prompt encoder stay frozen).  The decoder learns to produce accurate
# 256×256 lung masks from the encoder's [B,256,64,64] embeddings.
#
# What the pipeline learns:  With flip + brightness jitter augmentation, the
# decoder sees effectively 3× more training configurations per image.  It
# learns that the lung outline is symmetric (flip) and should be detected
# regardless of image contrast (brightness jitter).  This closes the ~0.06
# Dice gap to fully fine-tuned MedSAM reported in the literature.
#
# Prerequisite:  requires a component4_manifest.csv pointing to Montgomery
# and Shenzhen images + their manual lung masks.  The cell below auto-generates
# this manifest from the discovered dataset roots.
#
# Expected result:  Lung Dice improves from 0.832–0.874 to 0.89–0.93.
import subprocess, sys, os, csv
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
OUTPUT_C4 = Path('/kaggle/working/component4_runs')
OUTPUT_C4.mkdir(parents=True, exist_ok=True)

# ── Generate component4 manifest ──
MANIFEST_PATH = OUTPUT_C4 / 'component4_manifest.csv'

rows = []

def _add_montgomery(root):
    root = Path(root)
    for subdir in ['MontgomerySet/CXR_png', 'CXR_png', 'images']:
        img_dir = root / subdir
        if img_dir.exists():
            break
    else:
        img_dir = root
    for img in sorted(img_dir.rglob('*.png')):
        stem = img.stem
        left  = root / 'MontgomerySet' / 'ManualMask' / 'leftMask'  / f'{stem}.png'
        right = root / 'MontgomerySet' / 'ManualMask' / 'rightMask' / f'{stem}.png'
        if not left.exists():  # try alternate layout
            left  = root / 'ManualMask' / 'leftMask'  / f'{stem}.png'
            right = root / 'ManualMask' / 'rightMask' / f'{stem}.png'
        if left.exists() and right.exists():
            split = 'val' if hash(stem) % 5 == 0 else 'train'
            rows.append({'sample_id': stem, 'image_path': str(img),
                         'mask_path': f'{left}|{right}', 'dataset': 'montgomery', 'split': split})

def _add_shenzhen(root):
    root = Path(root)
    for subdir in ['ChinaSet_AllFiles/CXR_png', 'CXR_png', 'images']:
        img_dir = root / subdir
        if img_dir.exists():
            break
    else:
        img_dir = root
    mask_dir = root / 'ChinaSet_AllFiles' / 'Masks'
    if not mask_dir.exists():
        mask_dir = root / 'Masks'
    if not mask_dir.exists():
        print(f'  Shenzhen: no mask dir found, skipping'); return
    for img in sorted(img_dir.rglob('*.png')):
        stem = img.stem
        mask = mask_dir / f'{stem}_mask.png'
        if not mask.exists():
            mask = mask_dir / f'{stem}.png'
        if mask.exists():
            split = 'val' if hash(stem) % 5 == 0 else 'train'
            rows.append({'sample_id': stem, 'image_path': str(img),
                         'mask_path': str(mask), 'dataset': 'shenzhen', 'split': split})

if MONTGOMERY_ROOT:
    _add_montgomery(MONTGOMERY_ROOT)
    print(f'Montgomery: {sum(1 for r in rows if r["dataset"]=="montgomery")} samples')
if SHENZHEN_ROOT:
    _add_shenzhen(SHENZHEN_ROOT)
    print(f'Shenzhen: {sum(1 for r in rows if r["dataset"]=="shenzhen")} samples')

if not rows:
    print('WARNING: no lung mask annotations found — skipping Component 4 training')
else:
    with open(MANIFEST_PATH, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['sample_id','image_path','mask_path','dataset','split'])
        writer.writeheader(); writer.writerows(rows)
    print(f'Manifest written: {MANIFEST_PATH} ({len(rows)} rows)')

    env = os.environ.copy()
    env['COMPONENT4_TRAIN_MANIFEST'] = str(MANIFEST_PATH)
    env['COMPONENT4_VAL_MANIFEST']   = str(MANIFEST_PATH)
    env['COMPONENT4_SAVE_DIR']       = str(OUTPUT_C4)

    cmd = [
        sys.executable, '-m', 'src.training.train_component4_lung',
        '--config', str(REPO_DIR / 'configs' / 'component4_lung.yaml'),
    ]
    print('Starting MedSAM fine-tune with augmentation (Fix 4)...')
    result = subprocess.run(cmd, env=env, cwd=str(REPO_DIR))
    print(f'Component 4 exit code: {result.returncode}')
    c4_ckpt = OUTPUT_C4 / 'component4_mask_decoder.pt'
    print(f'Component 4 checkpoint: {"found" if c4_ckpt.exists() else "MISSING"}')


In [ ]:
# ── Cell 5: Fix 5 — MoE Cache Generation ───────────────────────────────────
# The expert decoders (C5) need per-image training targets: binary masks for
# each of the 4 pathology types (consolidation, cavity, fibrosis, nodule).
#
# What the pipeline learns to generate here:  For each training image we run
# the C1+C2 forward pass to get the image embedding and the TXV 7x7 feature
# map.  We then compute class-specific Grad-CAM maps for each expert's target
# pathology class and threshold them into pseudo-mask targets.
#
# For TBX11K images that have bounding box annotations, we use the GT boxes
# directly as consolidation/TB targets (stronger supervision than Grad-CAM).
#
# The cache is a directory of .pt files; each file contains:
#   image_emb:  [256, 64, 64]  — frozen C1 embedding
#   expert_masks: {expert_name: [1, 256, 256]}  — per-expert binary targets
#   expert_supervision_weights: {expert_name: float}  — 1.0 for GT, 0.5 for pseudo
import torch, json, os, sys
from pathlib import Path
import torch.nn.functional as F

REPO_DIR = Path('/kaggle/working/repo')
sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))

CACHE_DIR = Path('/kaggle/working/moe_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

from src.components.component1_encoder import Component1EncoderConfig, LoRAConfig, build_component1_encoder
from src.components.component1_dann import Component1DANNModel, DANNHead, DANNHeadConfig
from src.components.component2_txv import Component2SoftDomainContext, TXV_CLASS_NAMES
from src.data.harmonise import harmonise_sample
from src.core.device import pick_device
import numpy as np
from PIL import Image

device = pick_device()
print(f'Cache generation device: {device}')

# Load C1 (use freshly trained adapters if available)
c1_adapter = Path('/kaggle/working/component1_runs/component1_adapters.safetensors')
if not c1_adapter.exists():
    c1_adapter = REPO_DIR / 'checkpoints' / 'component1' / 'component1_adapters.safetensors'
print(f'C1 adapters: {c1_adapter}')

from src.components.component1_encoder import load_trainable_state_dict
encoder = build_component1_encoder(Component1EncoderConfig(
    backend='segment_anything',
    checkpoint_path=str(MEDSAM_CKPT),
    freeze_backbone=True,
    lora=LoRAConfig(rank=4, alpha=16.0, dropout=0.05, target_modules=('qkv',)),
)).to(device).eval()
if c1_adapter.exists():
    load_trainable_state_dict(encoder, str(c1_adapter))
    print('Loaded trained LoRA adapters')

# Load C2
c2_ckpt = Path('/kaggle/working/component2_runs/component2_routing_head.pt')
if not c2_ckpt.exists():
    c2_ckpt = None
c2_model = Component2SoftDomainContext(backend='auto').to(device).eval()
if c2_ckpt:
    c2_model.load_trained_routing_head(str(c2_ckpt))
    print(f'Loaded C2 routing head from {c2_ckpt}')

# Expert-to-TXV class index mapping
EXPERT_CLASS_MAP = {
    'consolidation': TXV_CLASS_NAMES.index('consolidation'),
    'fibrosis':      TXV_CLASS_NAMES.index('fibrosis'),
    'nodule':        TXV_CLASS_NAMES.index('nodule'),
    # cavity has no direct TXV class — use pneumonia + effusion as proxy
    'cavity':        TXV_CLASS_NAMES.index('pneumonia'),
}

PSEUDO_THRESHOLD = 0.45  # binarisation threshold for Grad-CAM pseudo-masks

def _cam_for_class(features_7x7, classifier_weight, class_idx, prob):
    """Return a [1, 256, 256] binary CAM for a single image."""
    w = classifier_weight[class_idx].view(-1, 1, 1)
    cam = torch.relu((features_7x7 * w).sum(0, keepdim=True)).unsqueeze(0)  # [1,1,7,7]
    cam = F.interpolate(cam, size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)
    max_val = cam.max()
    if max_val > 0:
        cam = cam / max_val
    cam = cam * float(prob)
    return (cam > PSEUDO_THRESHOLD).float()

def _load_tbx_bbox_masks(tbx_root, img_path):
    """Try to load TBX11K GT bounding boxes for this image as a [1,256,256] mask."""
    try:
        ann_dirs = list(Path(tbx_root).glob('annotations/json/*.json'))
        if not ann_dirs:
            return None
        stem = Path(img_path).stem
        for ann_file in ann_dirs:
            with open(ann_file) as f:
                ann = json.load(f)
            imgs = {i['id']: i for i in ann.get('images', [])}
            stem_map = {Path(i['file_name']).stem: i['id'] for i in ann.get('images', [])}
            if stem not in stem_map:
                continue
            img_id = stem_map[stem]
            img_info = imgs[img_id]
            W, H = img_info['width'], img_info['height']
            mask = torch.zeros(1, 256, 256)
            for ann_item in ann.get('annotations', []):
                if ann_item['image_id'] != img_id:
                    continue
                x, y, w, h = ann_item['bbox']
                x1 = int(x / W * 256); y1 = int(y / H * 256)
                x2 = int((x + w) / W * 256); y2 = int((y + h) / H * 256)
                mask[0, y1:y2, x1:x2] = 1.0
            if mask.sum() > 0:
                return mask
    except Exception:
        pass
    return None

# Collect images for cache generation (use TBX11K + small samples from other datasets)
from src.training.train_component1_dann import build_component1_manifest
all_samples = build_component1_manifest(
    paths_config=str(REPO_DIR / 'checkpoints' / 'component1' / 'paths.kaggle.yaml'),
    component1_config=str(REPO_DIR / 'checkpoints' / 'component1' / 'component1_dann.kaggle.yaml'),
)
# Limit: 500 per domain for cache speed
from collections import defaultdict
grouped = defaultdict(list)
for s in all_samples:
    grouped[s.dataset_id].append(s)
cache_samples = []
for ds, group in grouped.items():
    cache_samples.extend(group[:500])
print(f'Generating cache for {len(cache_samples)} images...')

from src.training.train_component1_dann import Component1DomainDataset
import io

n_written = 0
with torch.no_grad():
    for idx, sample in enumerate(cache_samples):
        try:
            ds_obj = Component1DomainDataset([sample], apply_augmentation=False)
            item = ds_obj[0]
            x3 = item['x_3ch'].unsqueeze(0).to(device)
            x224 = harmonise_sample({'image': np.array(Image.open(sample.image_path).convert('L')),
                                     'dataset_id': sample.dataset_id}).x_224.unsqueeze(0).to(device)

            img_emb, _ = encoder(x3, lambda_=0.0)
            txv_out = c2_model.forward_features(x224)
            feats = txv_out.features_7x7[0]  # [1024, 7, 7]
            probs = torch.sigmoid(txv_out.pathology_logits[0])
            cw = txv_out.classifier_weight

            expert_masks = {}
            sup_weights = {}

            # Try GT bbox for TBX11K consolidation expert
            if sample.dataset_id == 'tbx11k' and TBX11K_ROOT and sample.image_path:
                gt_mask = _load_tbx_bbox_masks(TBX11K_ROOT, sample.image_path)
                if gt_mask is not None:
                    expert_masks['consolidation'] = gt_mask
                    sup_weights['consolidation'] = 1.0

            for exp_name, cls_idx in EXPERT_CLASS_MAP.items():
                if exp_name in expert_masks:
                    continue
                if cw is not None:
                    pseudo = _cam_for_class(feats, cw, cls_idx, float(probs[cls_idx]))
                else:
                    pseudo = torch.zeros(1, 256, 256)
                expert_masks[exp_name] = pseudo.cpu()
                sup_weights[exp_name] = 0.5  # pseudo-labels are weaker

            payload = {
                'image_emb': img_emb[0].detach().cpu(),
                'expert_masks': expert_masks,
                'expert_supervision_weights': sup_weights,
                'dataset_id': sample.dataset_id,
            }
            out_path = CACHE_DIR / f'sample_{idx:05d}.pt'
            torch.save(payload, out_path)
            n_written += 1

        except Exception as e:
            if idx < 5:
                print(f'  sample {idx} skipped: {e}')
            continue

        if (idx + 1) % 100 == 0:
            print(f'  {idx+1}/{len(cache_samples)} processed, {n_written} written')

print(f'Cache generation complete: {n_written} files in {CACHE_DIR}')


In [ ]:
# ── Cell 6: Fix 5 — Train MoE Expert Decoders ──────────────────────────────
# What this trains:  4 lightweight expert decoders (C5), one per pathology:
#   consolidation, cavity, fibrosis, nodule.
# Each expert is a small U-Net-style decoder that takes the C1 embedding
# [256, 64, 64] and outputs a [1, 256, 256] lesion probability map.
#
# What the pipeline learns:  Each expert specialises.  The consolidation
# expert learns to produce dense, solid-region masks.  The nodule expert
# learns to produce small, round, high-confidence blobs.  The routing gate
# (C3) then decides how much weight each expert's output gets per patient.
#
# Prerequisite: Cell 5 must complete (CACHE_DIR must be non-empty).
import subprocess, sys, json
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
CACHE_DIR = Path('/kaggle/working/moe_cache')
OUTPUT_MOE = Path('/kaggle/working/moe_runs')
OUTPUT_MOE.mkdir(parents=True, exist_ok=True)

cache_files = list(CACHE_DIR.glob('*.pt'))
print(f'Cache files available: {len(cache_files)}')

if len(cache_files) < 10:
    print('WARNING: fewer than 10 cache files — expert training will use synthetic data')

# Write a temporary patched config with Kaggle paths
import yaml
moe_cfg_path = REPO_DIR / 'configs' / 'moe.yaml'
with open(moe_cfg_path) as f:
    moe_cfg = yaml.safe_load(f)

moe_cfg['moe_training']['save_dir'] = str(OUTPUT_MOE)
moe_cfg['moe_training']['pretrain']['epochs'] = 10
moe_cfg['moe_training']['pretrain']['batch_size'] = 8
moe_cfg['moe_training']['num_workers'] = 2
moe_cfg['moe_training']['amp'] = True

tmp_moe_cfg = OUTPUT_MOE / 'moe_kaggle.yaml'
with open(tmp_moe_cfg, 'w') as f:
    yaml.dump(moe_cfg, f)

for expert in ['consolidation', 'cavity', 'fibrosis', 'nodule']:
    print(f'\n── Training expert: {expert} ──')
    cmd = [
        sys.executable, '-m', 'src.training.train_experts',
        '--config', str(tmp_moe_cfg),
        '--expert', expert,
        '--cache-dir', str(CACHE_DIR),
    ]
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    print(f'{expert} exit code: {result.returncode}')

print('\nExpert checkpoints:')
for p in sorted(OUTPUT_MOE.glob('expert_*.pt')):
    print(f'  {p.name}: {p.stat().st_size // 1024} KB')


In [ ]:
# ── Cell 7: Collect and summarise all outputs ───────────────────────────────
import shutil
from pathlib import Path

FINAL = Path('/kaggle/working/all_checkpoints')
FINAL.mkdir(parents=True, exist_ok=True)

copies = [
    (Path('/kaggle/working/component1_runs/component1_dann_full.pt'),    FINAL / 'component1_dann_full.pt'),
    (Path('/kaggle/working/component1_runs/component1_adapters.safetensors'), FINAL / 'component1_adapters.safetensors'),
    (Path('/kaggle/working/component2_runs/component2_routing_head.pt'), FINAL / 'component2_routing_head.pt'),
    (Path('/kaggle/working/component4_runs/component4_mask_decoder.pt'), FINAL / 'component4_mask_decoder.pt'),
]

# Expert checkpoints
for p in Path('/kaggle/working/moe_runs').glob('expert_*.pt'):
    copies.append((p, FINAL / p.name))

print('Checkpoint summary:')
print(f'{"File":<45} {"Size":>10}')
print('-' * 57)
for src, dst in copies:
    if src.exists():
        shutil.copy2(src, dst)
        size_kb = dst.stat().st_size // 1024
        print(f'{dst.name:<45} {size_kb:>8} KB')
    else:
        print(f'{src.name:<45} {"MISSING":>10}')

print(f'\nAll outputs are in: {FINAL}')
print('Download /kaggle/working/all_checkpoints/ and place the files in:')
print('  component1_dann_full.pt         → checkpoints/component1/')
print('  component1_adapters.safetensors → checkpoints/component1/')
print('  component2_routing_head.pt      → checkpoints/component2/')
print('  component4_mask_decoder.pt      → checkpoints/component4/')
print('  expert_*.pt                     → checkpoints/component_moe/')
